In [ ]:
#r "nuget: Microsoft.CognitiveServices.Speech, 1.19.0"
#r "nuget: Azure.Identity, 1.12.0"


In [ ]:
using System;
using System.IO;
using Azure.Core;
using Azure.Identity;
using System.Threading.Tasks;
using Microsoft.CognitiveServices.Speech;
using Microsoft.CognitiveServices.Speech.Audio;


In [ ]:
async Task<string> TranscribeAudioAsync(string endpoint, string audioFilePath)
{
    var credential = new DefaultAzureCredential();
    var tokenRequestContext = new TokenRequestContext(new[] { "https://cognitiveservices.azure.com/.default" });
    var accessToken = await credential.GetTokenAsync(tokenRequestContext);

    var resourceId = "a resource id";
    var authorizationToken = $"aad#{resourceId}#{accessToken.Token}";
    
    var config = SpeechConfig.FromAuthorizationToken(authorizationToken,"WestEurope");

    using var audioInput = AudioConfig.FromWavFileInput(audioFilePath);
    using var recognizer = new SpeechRecognizer(config, audioInput);

    Console.WriteLine("Transcribing...");

    var result = await recognizer.RecognizeOnceAsync();

    if (result.Reason == ResultReason.RecognizedSpeech)
    {
        return $"Transcription: {result.Text}";
    }
    else if (result.Reason == ResultReason.NoMatch)
    {
        return "No speech could be recognized.";
    }
else if (result.Reason == ResultReason.Canceled)
    {
        var cancellation = CancellationDetails.FromResult(result);
        var errorDetails = $"CANCELED: Reason={cancellation.Reason}\n";

        if (cancellation.Reason == CancellationReason.Error)
        {
            errorDetails += $"CANCELED: ErrorCode={cancellation.ErrorCode}\n";
            errorDetails += $"CANCELED: ErrorDetails={cancellation.ErrorDetails}\n";
            
        }

        return errorDetails;
    }

    return "An unexpected error occurred.";
}


In [ ]:
async Task<string> TranscribeAudioAsync(string subscriptionKey, string region, string audioFilePath)
{
    var config = SpeechConfig.FromSubscription(subscriptionKey, region);
    using var audioInput = AudioConfig.FromWavFileInput(audioFilePath);
    using var recognizer = new SpeechRecognizer(config, audioInput);

    var transcription = new StringBuilder();
    var taskCompletionSource = new TaskCompletionSource<string>();

    recognizer.Recognized += (s, e) =>
    {
        if (e.Result.Reason == ResultReason.RecognizedSpeech)
        {
            transcription.Append(e.Result.Text);
            transcription.Append(" ");
        }
        else if (e.Result.Reason == ResultReason.NoMatch)
        {
            Console.WriteLine("No speech could be recognized.");
        }
    };

    recognizer.Canceled += (s, e) =>
    {
        Console.WriteLine($"CANCELED: Reason={e.Reason}");

        if (e.Reason == CancellationReason.Error)
        {
            Console.WriteLine($"CANCELED: ErrorCode={e.ErrorCode}");
            Console.WriteLine($"CANCELED: ErrorDetails={e.ErrorDetails}");
            Console.WriteLine("CANCELED: Did you update the subscription info?");
        }

        taskCompletionSource.TrySetResult(transcription.ToString().Trim());
    };

    recognizer.SessionStopped += (s, e) =>
    {
        Console.WriteLine("Session stopped.");
        taskCompletionSource.TrySetResult(transcription.ToString().Trim());
    };

    await recognizer.StartContinuousRecognitionAsync().ConfigureAwait(false);

    // Wait for the session to stop or cancel
    var result = await taskCompletionSource.Task;

    await recognizer.StopContinuousRecognitionAsync().ConfigureAwait(false);

    return result;
}


In [ ]:
async Task<string> TranscribeAudioFromStreamAsync(string subscriptionKey, string region, string audioFilePath)
{
    var config = SpeechConfig.FromSubscription(subscriptionKey, region);

    // Open the local WAV file as a stream
    using FileStream audioStream = File.OpenRead(audioFilePath);
    var pushStream = AudioInputStream.CreatePushStream();

    // Read audio data from file and push it to the push stream
    byte[] buffer = new byte[1024];
    int bytesRead;
    while ((bytesRead = await audioStream.ReadAsync(buffer, 0, buffer.Length)) > 0)
    {
        pushStream.Write(buffer, bytesRead);
    }
    pushStream.Close();

    using var audioInput = AudioConfig.FromStreamInput(pushStream);
    using var recognizer = new SpeechRecognizer(config, audioInput);

    var transcription = new StringBuilder();
    var taskCompletionSource = new TaskCompletionSource<string>();

    recognizer.Recognized += (s, e) =>
    {
        if (e.Result.Reason == ResultReason.RecognizedSpeech)
        {
            transcription.Append(e.Result.Text);
            transcription.Append(" ");
        }
        else if (e.Result.Reason == ResultReason.NoMatch)
        {
            Console.WriteLine("No speech could be recognized.");
        }
    };

    recognizer.Canceled += (s, e) =>
    {
        Console.WriteLine($"CANCELED: Reason={e.Reason}");

        if (e.Reason == CancellationReason.Error)
        {
            Console.WriteLine($"CANCELED: ErrorCode={e.ErrorCode}");
            Console.WriteLine($"CANCELED: ErrorDetails={e.ErrorDetails}");
         
        }

        taskCompletionSource.TrySetResult(transcription.ToString().Trim());
    };

    recognizer.SessionStopped += (s, e) =>
    {
        Console.WriteLine("Session stopped.");
        taskCompletionSource.TrySetResult(transcription.ToString().Trim());
    };

    await recognizer.StartContinuousRecognitionAsync().ConfigureAwait(false);

    // Wait for the session to stop or cancel
    var result = await taskCompletionSource.Task;

    await recognizer.StopContinuousRecognitionAsync().ConfigureAwait(false);

    return result;
}


In [ ]:

string endpoint = "https://westeurope.stt.speech.microsoft.com/";
string audioFilePath = "/Users/yoavdobrin/workspace/fta/event_processing/src/DurableOrchestrator/tests/sample_call_audio.wav";

var res = await TranscribeAudioAsync(endpoint, audioFilePath);

Console.WriteLine(res);